# ATM Withdrawals as an OLAP Store

Packs batches of withdrawals into fixed-width blobs, stores one blob per key in a UStore
collection, reads them back and aggregates them on the GPU.
The helpers come from `banks_atm_olap.py`, so the notebook and the script stay in step.

cuxfilter reached end of life with RAPIDS 26.06, its final release, so the whole stack pins
there; create the environment from `banks_atm_olap.yml`.

In [ ]:
import cudf
import cuxfilter as cux
import pandas as pd
import ustore.smashtable as ustore
from cuxfilter import charts

from banks_atm_olap import (
    GEOJSON_LINK,
    MAPBOX_API_KEY,
    load_measurements,
    load_zip_codes,
    rolling_mean_amounts,
    store_batches,
)

In [ ]:
zip_codes = load_zip_codes()

database = ustore.DataBase()
measurements = database.main
store_batches(measurements, zip_codes)

recovered = load_measurements(measurements)
recovered["time"] = pd.to_datetime(recovered["timestamps"], unit="s")
recovered.head()

In [ ]:
smoothed = rolling_mean_amounts(recovered["amounts"])
print(f"Recovered {len(recovered)} withdrawals, smoothed into {len(smoothed)} windowed means")

cux_frame = cux.DataFrame.from_dataframe(cudf.from_pandas(recovered))

In [ ]:
chart_map = charts.choropleth(
    x="zips",
    elevation_column="humans",
    elevation_aggregate_fn="mean",
    color_column="amounts",
    color_aggregate_fn="mean",
    mapbox_api_key=MAPBOX_API_KEY,
    geoJSONSource=GEOJSON_LINK,
    title="ATM Withdrawals Across the US",
)
chart_humans = charts.bar("humans")
slider_time = charts.date_range_slider("time", title="Timeframe")
overall_amounts = charts.number(
    expression="amounts",
    aggregate_fn="sum",
    format="${value:,.1f}",
    title="Total Withdrawals",
)
overall_humans = charts.number(
    expression="humans",
    aggregate_fn="sum",
    format="{value:,.0f}",
    title="Humans Detected",
)

In [ ]:
dashboard = cux_frame.dashboard(
    [chart_map, chart_humans],
    layout_array=[[1, 2]],
    sidebar=[slider_time, overall_amounts, overall_humans],
    theme=cux.themes.rapids,
    title="Map of ATMs",
)

dashboard.app()